# Marine AI Decision Pipeline

**Author: Feras Elghandour**

Integrate tabular prediction, targeted generative augmentation, and traceable decision rules in a single abalone analysis pipeline.

A random-forest ring-count predictor, a VAE trained on older training specimens, synthetic augmentation, explicit defer/protect/allow rules, and an in-memory decision trace.

Read README.md for result scope and DATA.md for input preparation. Run cells in order in a clean environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import json, datetime, math
from collections import Counter

np.random.seed(42)
torch.manual_seed(42)

print(f"NumPy: {np.__version__}  |  Pandas: {pd.__version__}  |  PyTorch: {torch.__version__}")
print("All imports successful.")

In [ ]:
def load_and_clean(path: str) -> pd.DataFrame:
    """
    Stage 1: Load and clean the abalone dataset.
    Applies P1 workflow patterns: type checking, missing value report,
    derived features (age group bins, encoded sex).
    """
    df = pd.read_csv(path)
    print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Missing values: {df.isnull().sum().sum()}")
    print(f"Dtypes: {df.dtypes.value_counts().to_dict()}")

    # Encode categorical Sex feature
    le = LabelEncoder()
    df['Sex_enc'] = le.fit_transform(df['Sex'])  # F=0, I=1, M=2

    # Derive conservation-relevant age group bins (rings)
    df['Age_group'] = pd.cut(df['Rings'], bins=[0, 8, 15, 30],
                             labels=['Young (1-8)', 'Mid (9-15)', 'Old (16+)'])
    return df


df = load_and_clean('abalone.csv')

print("\nAge group distribution (key for conservation):")
print(df['Age_group'].value_counts().sort_index())
print("\nNote: Old specimens (261 samples, 6.2%) are severely underrepresented → motivates VAE augmentation")
df.head(3)

In [ ]:
FEATURES = ['Length', 'Diameter', 'Height', 'Whole_weight',
            'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Sex_enc']
TARGET   = 'Rings'

X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

train_ids, test_ids = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X[train_ids])
X_scaled = scaler.transform(X)
X_tr, X_te = X_scaled[train_ids], X_scaled[test_ids]
y_tr, y_te = y[train_ids], y[test_ids]
assert not set(train_ids) & set(test_ids)

print("Training baseline Random Forest (200 trees)...")
rf_base = RandomForestRegressor(n_estimators=200, max_depth=15,
                                 min_samples_leaf=3, random_state=42, n_jobs=2)
rf_base.fit(X_tr, y_tr)
y_pred_base = rf_base.predict(X_te)

mae_base = round(mean_absolute_error(y_te, y_pred_base), 4)
r2_base  = round(r2_score(y_te, y_pred_base), 4)
old_mask = y_te > 15
mae_old_base = round(mean_absolute_error(y_te[old_mask], y_pred_base[old_mask]), 4)

print(f"\nBaseline RF — Overall MAE: {mae_base}  R²: {r2_base}")
print(f"Old-class MAE (>15 rings): {mae_old_base}  ← target for VAE augmentation")
print(f"Old specimens in test set: {old_mask.sum()} / {len(y_te)}")

In [ ]:
class VAE(nn.Module):
    """
    Variational Autoencoder for tabular feature augmentation.
    Reuses the tabular VAE architecture from the related generative experiment.
    Trained on old-class specimens only to generate targeted augmentation samples.
    """
    def __init__(self, input_dim=8, hidden=64, latent=4):
        super().__init__()
        self.enc1    = nn.Linear(input_dim, hidden)
        self.enc2    = nn.Linear(hidden, hidden)
        self.fc_mu   = nn.Linear(hidden, latent)
        self.fc_lv   = nn.Linear(hidden, latent)
        self.dec1    = nn.Linear(latent, hidden)
        self.dec2    = nn.Linear(hidden, hidden)
        self.dec_out = nn.Linear(hidden, input_dim)

    def encode(self, x):
        h = torch.relu(self.enc1(x))
        return self.fc_mu(torch.relu(self.enc2(h))), self.fc_lv(torch.relu(self.enc2(h)))

    def reparameterize(self, mu, lv):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * lv)

    def decode(self, z):
        h = torch.relu(self.dec1(z))
        return self.dec_out(torch.relu(self.dec2(h)))

    def forward(self, x):
        mu, lv = self.encode(x)
        return self.decode(self.reparameterize(mu, lv)), mu, lv


# Train VAE on OLD specimens only
old_mask_full = (df['Age_group'] == 'Old (16+)') & df.index.isin(train_ids)
X_old = torch.tensor(X_scaled[old_mask_full.values], dtype=torch.float32)
print(f"Old specimens for VAE training: {len(X_old)} samples")

vae = VAE(input_dim=8, hidden=64, latent=4)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(100):
    vae.train()
    recon, mu, lv = vae(X_old)
    recon_loss = nn.functional.mse_loss(recon, X_old, reduction='sum')
    kl_loss    = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp())
    loss = (recon_loss + kl_loss) / len(X_old)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1}/100 — ELBO loss: {loss.item():.4f}")

# Generate 200 synthetic old specimens
vae.eval()
with torch.no_grad():
    z_sample   = torch.randn(200, 4)
    synth_feats = vae.decode(z_sample).numpy()

old_rings_mean = float(df[old_mask_full]['Rings'].mean())
old_rings_std  = float(df[old_mask_full]['Rings'].std())
synth_rings = np.clip(
    np.random.normal(old_rings_mean, old_rings_std, 200), 16, 30
).astype(np.float32)

print(f"\nGenerated 200 synthetic Old specimens")
print(f"Synthetic rings — mean: {synth_rings.mean():.2f}, std: {synth_rings.std():.2f}")
print(f"Real old rings  — mean: {old_rings_mean:.2f}, std: {old_rings_std:.2f}")

In [ ]:
# Train augmented RF and compare
X_aug = np.vstack([X_tr, synth_feats])
y_aug = np.concatenate([y_tr, synth_rings])

rf_aug = RandomForestRegressor(n_estimators=200, max_depth=15,
                                min_samples_leaf=3, random_state=42, n_jobs=2)
rf_aug.fit(X_aug, y_aug)
y_pred_aug = rf_aug.predict(X_te)

mae_aug     = round(mean_absolute_error(y_te, y_pred_aug), 4)
r2_aug      = round(r2_score(y_te, y_pred_aug), 4)
mae_old_aug = round(mean_absolute_error(y_te[old_mask], y_pred_aug[old_mask]), 4)

print("=== Augmentation Experiment Results ===")
comparison = pd.DataFrame({
    'Model':          ['Baseline RF', 'Augmented RF (+200 synthetic old)'],
    'Overall MAE':    [mae_base, mae_aug],
    'Overall R²':     [r2_base, r2_aug],
    'Old-class MAE':  [mae_old_base, mae_old_aug],
    'Training Size':  [len(X_tr), len(X_aug)],
})
print(comparison.to_string(index=False))
print(f"\nOld-class MAE improvement: {mae_old_base} → {mae_old_aug} "
      f"({((mae_old_base-mae_old_aug)/mae_old_base*100):.1f}% reduction)")

In [ ]:
HARVEST_MIN_RINGS           = 11    # minimum rings required for harvest approval
CONFIDENCE_THRESHOLD        = 0.80  # minimum prediction confidence required

TOOL_REGISTRY = {}
def register_tool(fn): TOOL_REGISTRY[fn.__name__] = fn; return fn


@register_tool
def predict_age(measurements: str) -> str:
    """
    Tool 1: Predict abalone age (ring count) from physical measurements.
    Uses the VAE-augmented Random Forest model (Stage 2+3).
    Returns predicted rings, uncalibrated tree-agreement score, and approximate age in years.
    """
    parts = [float(x) for x in measurements.split(',')]
    x_sc  = scaler.transform(np.array([parts]))
    pred  = rf_aug.predict(x_sc)[0]
    # Confidence: 1 - CV of individual tree predictions
    tree_preds = np.array([t.predict(x_sc)[0] for t in rf_aug.estimators_])
    conf = float(np.clip(1 - (tree_preds.std() / (tree_preds.mean() + 1e-6)), 0, 1))
    return json.dumps({"predicted_rings": round(float(pred), 2),
                       "confidence":      round(conf, 3),
                       "age_years":        round(float(pred) + 1.5, 1)})


@register_tool
def harvest_decision(pred_json: str) -> str:
    """
    Tool 2: Apply conservation rules to a prediction result.
    Returns HARVEST_OK, PROTECT, or DEFER with justification.
    """
    d     = json.loads(pred_json)
    rings = d['predicted_rings']
    conf  = d['confidence']
    if conf < CONFIDENCE_THRESHOLD:
        return f"DEFER: Confidence {conf:.2f} below threshold {CONFIDENCE_THRESHOLD}. Recommend physical re-measurement."
    if rings < HARVEST_MIN_RINGS:
        return f"PROTECT: Predicted {rings:.1f} rings < minimum {HARVEST_MIN_RINGS}. Do not harvest."
    return f"HARVEST_OK: Predicted {rings:.1f} rings ≥ minimum {HARVEST_MIN_RINGS}. Approved."


@register_tool
def log_decision(entry: str) -> str:
    """Tool 3: Append a decision to the conservation audit log."""
    return f"[{datetime.datetime.now().strftime('%H:%M:%S')}] LOGGED: {entry[:80]}"


class ConservationAgent:
    """
    ReAct-pattern agent for abalone harvest governance.
    Composes explicit tools and decision rules.

    Safeguards (from P6):
        - Confidence threshold: defers uncertain predictions
        - Minimum rings rule: hard-coded conservation constraint
        - In-memory event log: records the decision sequence
        - Working memory: caches predictions to avoid duplicate calls
    """
    def __init__(self):
        self.audit        = []          # append-only event log
        self.memory       = {}          # working memory cache
        self.decision_log = []          # conservation record
        self.counts       = Counter()

    def _log(self, event, msg):
        self.audit.append({'ts': datetime.datetime.now().strftime('%H:%M:%S'),
                           'event': event, 'msg': str(msg)[:120]})

    def evaluate(self, specimen_id: str, measurements: str) -> dict:
        """Run one specimen through the full Thought → Act → Decide cycle."""
        self._log('INPUT', f"ID={specimen_id} meas={measurements[:50]}")

        # THINK: check cache
        if measurements in self.memory:
            pred_result = self.memory[measurements]
            self._log('CACHE_HIT', measurements[:50])
        else:
            pred_result = predict_age(measurements)
            self.memory[measurements] = pred_result
            self._log('PREDICT', pred_result)

        # ACT: apply decision rule
        decision = harvest_decision(pred_result)
        self._log('DECIDE', decision)

        # OBSERVE: log and record
        log_msg = log_decision(f"ID={specimen_id}: {decision}")
        self._log('LOG', log_msg)

        decision_key = decision.split(':')[0]
        self.counts[decision_key] += 1
        result = {'id': specimen_id, 'prediction': json.loads(pred_result),
                  'decision': decision, 'decision_key': decision_key}
        self.decision_log.append(result)
        return result


agent = ConservationAgent()
print(f"Conservation Agent initialized.")
print(f"Tools: {list(TOOL_REGISTRY.keys())}")
print(f"Harvest minimum: {HARVEST_MIN_RINGS} rings  |  Confidence threshold: {CONFIDENCE_THRESHOLD}")

In [ ]:
# Run 10 real test specimens through the agent
print(f"{'='*72}")
print(f"CONSERVATION AGENT — HARVEST EVALUATION SESSION")
print(f"{'='*72}")

eval_results = []
for i in range(10):
    raw = scaler.inverse_transform(X_te[i:i+1])[0]
    mstr = ','.join([str(round(v, 4)) for v in raw])
    result = agent.evaluate(f"SPEC-{i+1:03d}", mstr)
    actual = y_te[i]
    result['actual_rings'] = actual
    eval_results.append(result)

    icon = {'HARVEST_OK': '✓', 'PROTECT': '🛡', 'DEFER': '⚠'}.get(result['decision_key'], '?')
    print(f"  {icon} SPEC-{i+1:03d} | Actual={actual:.0f}r | "
          f"Pred={result['prediction']['predicted_rings']:.1f}r | "
          f"Conf={result['prediction']['confidence']:.2f} | "
          f"{result['decision'][:55]}")

print(f"\nDecision summary: {dict(agent.counts)}")

In [ ]:
# Print audit log
print("AUDIT LOG (last 15 entries):")
print(f"  {'Time':>9}  {'Event':>12}  Content")
print('-' * 80)
for e in agent.audit[-15:]:
    print(f"  {e['ts']:>9}  {e['event']:>12}  {e['msg'][:58]}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Fig A: Feature Importance (P3 insight reused)
importances = rf_aug.feature_importances_
feat_labels = ['Length', 'Diameter', 'Height', 'Whole wt',
               'Shucked wt', 'Viscera wt', 'Shell wt', 'Sex']
sorted_idx = np.argsort(importances)
axes[0,0].barh([feat_labels[i] for i in sorted_idx],
                importances[sorted_idx], color='steelblue', alpha=0.85)
axes[0,0].set_title('Fig A — Feature Importance (Augmented RF)', fontweight='bold')
axes[0,0].set_xlabel('Importance Score')
axes[0,0].grid(axis='x', alpha=0.3)

# Fig B: Actual vs Predicted
colors_scatter = ['crimson' if v > 15 else 'steelblue' for v in y_te]
axes[0,1].scatter(y_te, y_pred_aug, c=colors_scatter, alpha=0.4, s=12)
axes[0,1].plot([0, 30], [0, 30], 'k--', lw=1)
axes[0,1].set_title(f'Fig B — Actual vs Predicted Rings (MAE={mae_aug})', fontweight='bold')
axes[0,1].set_xlabel('Actual Rings'); axes[0,1].set_ylabel('Predicted Rings')
p1 = mpatches.Patch(color='steelblue', label='Young/Mid (≤15 rings)')
p2 = mpatches.Patch(color='crimson',   label='Old (>15 rings)')
axes[0,1].legend(handles=[p1, p2], fontsize=8)

# Fig C: VAE Old-class distribution
axes[1,0].hist(X_old.numpy()[:, 6], bins=20, alpha=0.55,
               color='steelblue', density=True, label='Real Old Specimens')
axes[1,0].hist(synth_feats[:, 6], bins=20, alpha=0.55,
               color='darkorange', density=True, label='VAE Synthetic Specimens')
axes[1,0].set_title('Fig C — Real vs VAE-Synthetic Old Class (Shell Weight)', fontweight='bold')
axes[1,0].set_xlabel('Standardized Shell Weight'); axes[1,0].set_ylabel('Density')
axes[1,0].legend(); axes[1,0].grid(alpha=0.2)

# Fig D: Agent decisions
dec_counts = dict(agent.counts)
color_map = {'HARVEST_OK': '#2ecc71', 'PROTECT': '#e74c3c', 'DEFER': '#f39c12'}
axes[1,1].bar(dec_counts.keys(), dec_counts.values(),
              color=[color_map.get(k, 'gray') for k in dec_counts.keys()],
              alpha=0.85, edgecolor='white', linewidth=1.5)
axes[1,1].set_title('Fig D — Agent Decision Distribution (10 specimens)', fontweight='bold')
axes[1,1].set_ylabel('Count'); axes[1,1].grid(axis='y', alpha=0.3)
for k, v in dec_counts.items():
    axes[1,1].text(list(dec_counts.keys()).index(k), v + 0.05,
                   str(v), ha='center', fontweight='bold', fontsize=11)

plt.suptitle('Marine Conservation AI Pipeline — Integrated Results\n'
             '(Projects 1, 3, 5, 6 | Abalone Dataset | n=4,177)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_pipeline_results.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Architecture diagram
fig2, ax = plt.subplots(figsize=(14, 4.5))
ax.axis('off')

stages = [
    (1.0, 'Stage 1\nData Ingestion\n& Cleaning\n(P1)', '#4a90d9'),
    (3.5, 'Stage 2\nRF Age\nPredictor\n(P3 ML)', '#2ecc71'),
    (6.0, 'Stage 3\nVAE Data\nAugmentation\n(P5 GenAI)', '#9b59b6'),
    (8.5, 'Stage 4\nConservation\nAgent\n(P6 Agentic)', '#e67e22'),
    (11.0, 'Output\nHarvest\nDecisions\n+ Audit Log', '#1abc9c'),
]
for x, label, color in stages:
    ax.add_patch(plt.Rectangle((x-0.9, 1.5), 1.8, 2.0,
                                facecolor=color, alpha=0.85,
                                edgecolor='white', linewidth=2, zorder=2))
    ax.text(x, 2.5, label, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white', zorder=3)

for i in range(len(stages) - 1):
    x1, x2 = stages[i][0] + 0.9, stages[i+1][0] - 0.9
    ax.annotate('', xy=(x2, 2.5), xytext=(x1, 2.5),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))

labels_b = ['abalone.csv\n4177 rows', 'MAE and R²\nsee executed results',
            '200 synthetic\nOld specimens', '10 evaluations\nHARVEST/PROTECT/DEFER',
            'Decision log\n+ audit trail']
for (x, _, _), lbl in zip(stages, labels_b):
    ax.text(x, 1.35, lbl, ha='center', va='top', fontsize=7.5, color='#555')

ax.set_xlim(0, 12.5); ax.set_ylim(0.5, 4.5)
ax.set_title('Architecture Diagram — Marine Conservation AI Pipeline (Projects P1 · P3 · P5 · P6)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_architecture.png', dpi=120, bbox_inches='tight')
plt.show()